# 语义通信收发机 v2 — 完整训练笔记本

这个笔记本包含训练所需的**全部代码**。每个 `%%writefile` 单元格会把一个代码文件写到 Colab 机器的 `/content/sc/` 下，然后由最后的单元格调用。

**使用方法：** 先在菜单“代码执行程序 → 更改运行时类型”里选择 GPU，然后点“代码执行程序 → 全部运行”，或者从上到下依次运行。

**Drive 里的项目文件夹** `MyDrive/semantic-comm-transceiver/`：

| 路径 | 内容 |
|---|---|
| `data/` | 数据压缩包分片（第 2 节还原为 4 个 pkl） |
| `checkpoints/` | 模型权重、评估结果、`run_all.log` 日志（运行时生成） |
| `results/` | 汇总表 `summary.md` 和曲线图（运行时生成） |

**断线后：** 重新“全部运行”即可。已训练完的模型会跳过，训练到一半的会从 `last.pt` 继续，已有的评估结果不会重算。

**实验内容：** 8 个模型 × 80 epoch。AWGN 信道：Baseline、Improved 和 3 个消融版本（去掉 FiLM / 去掉带掩码的功率归一化 / 去掉特征去噪损失）；Rayleigh 信道：Baseline、Improved 和 1 个消融版本（去掉接收端有效 SNR）。T4 上训练约 16 小时，A100/L4 会快很多。

## 1. 准备工作目录、挂载 Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJ = '/content/drive/MyDrive/semantic-comm-transceiver'
os.makedirs('/content/sc/models', exist_ok=True)
os.makedirs('/content/sc/data', exist_ok=True)
%cd /content/sc

## 2. 代码文件

下面每个单元格写出一个文件。改进相关的核心代码在 `models/` 下的几个文件里。

### `requirements.txt` — 依赖列表

需要安装的 Python 包。

In [ ]:
%%writefile requirements.txt
torch>=2.0.0
numpy>=1.24.0
nltk>=3.8
transformers>=4.30.0
datasets>=2.14.0
tqdm>=4.65.0
matplotlib>=3.7.0
scikit-learn>=1.3.0
bert-score>=0.3.12

### `config.py` — 配置

所有超参数、路径、评估 SNR 点、验证集比例、BERT 融合调参网格。

In [ ]:
%%writefile config.py
import os
import torch


class Config:
    # ----- paths -----
    base_dir = os.path.dirname(os.path.abspath(__file__))
    data_dir = os.path.join(base_dir, "data")
    # SC_CKPT_DIR / SC_RESULT_DIR let cloud runs write to persistent storage
    checkpoint_dir = os.environ.get("SC_CKPT_DIR", os.path.join(base_dir, "checkpoints"))
    result_dir = os.environ.get("SC_RESULT_DIR", os.path.join(base_dir, "results"))
    bert_dir = os.path.join(checkpoint_dir, "bert_europarl")

    # ----- vocabulary & tokens -----
    PAD_TOKEN = "<PAD>"
    START_TOKEN = "<START>"
    END_TOKEN = "<END>"
    UNK_TOKEN = "<UNK>"
    PAD_IDX = 0
    START_IDX = 1
    END_IDX = 2
    UNK_IDX = 3
    vocab_size = 10000
    max_seq_len = 32
    min_seq_len = 4

    # ----- model architecture -----
    d_model = 128
    n_heads = 8
    d_ff = 512
    n_encoder_layers = 4
    n_decoder_layers = 4
    channel_dim = 16
    dropout = 0.1

    # ----- training -----
    batch_size = 256
    epochs = 80
    lr = 1e-3
    lr_min = 1e-5
    weight_decay = 1e-5
    grad_clip = 1.0
    snr_train_low = -10.0   # dB
    snr_train_high = 20.0   # dB
    warmup_epochs = 5
    val_fraction = 0.05     # carved out of the training split
    val_snr_list = [-5.0, 0.0, 5.0]   # model selection SNRs (dB)

    # ----- composite loss weights -----
    alpha_ce = 1.0
    beta_semantic = 0.1     # feature-denoising loss weight (improved model only)

    # ----- evaluation SNR points -----
    eval_snr_list = [-10, -5, 0, 5, 10, 15, 20]
    eval_max_samples = 10000

    # ----- LLM fusion refiner (tuned on the validation split) -----
    llm_topk = 5
    llm_lambda_grid = [0.0, 0.25, 0.5, 1.0, 2.0]
    llm_tau_grid = [0.5, 0.7, 0.9]
    llm_tune_snrs = [-5.0, 0.0, 5.0]
    llm_tune_samples = 2000

    # ----- channel types -----
    AWGN = "awgn"
    RAYLEIGH = "rayleigh"

    # ----- device -----
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    @classmethod
    def ensure_dirs(cls):
        for d in [cls.data_dir, cls.checkpoint_dir, cls.result_dir]:
            os.makedirs(d, exist_ok=True)

### `utils.py` — 工具函数

随机种子、token id 转文本、混合精度类型选择。

In [ ]:
%%writefile utils.py
import math
import random
import numpy as np
import torch


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def snr_db_to_noise_std(snr_db: float) -> float:
    """Convert SNR in dB to noise standard deviation (assuming unit signal power)."""
    snr_linear = 10 ** (snr_db / 10.0)
    return math.sqrt(1.0 / snr_linear)


def ids_to_text(ids, vocab_inv, end_idx=2):
    """Convert a 1-D tensor of token ids back to a string."""
    tokens = []
    for idx in ids:
        idx = idx.item() if hasattr(idx, "item") else int(idx)
        if idx == end_idx:
            break
        word = vocab_inv.get(idx, "")
        if word and word not in ("<PAD>", "<START>"):
            tokens.append(word)
    return " ".join(tokens)


def batch_ids_to_texts(batch_ids, vocab_inv, end_idx=2):
    """Convert a batch of token-id sequences to a list of strings."""
    return [ids_to_text(seq, vocab_inv, end_idx) for seq in batch_ids]


class AverageMeter:
    """Track running averages."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def amp_dtype(device):
    """Mixed-precision dtype for CUDA: bf16 on Ampere+, fp16 otherwise."""
    if device.type != "cuda":
        return None
    major, _ = torch.cuda.get_device_capability(device)
    return torch.bfloat16 if major >= 8 else torch.float16

### `dataset.py` — 数据加载

读取 pkl 数据；从训练集固定划出 5% 作为验证集。

In [ ]:
%%writefile dataset.py
import os
import pickle

import torch
from torch.utils.data import Dataset, DataLoader

from config import Config


class EuroparlDataset(Dataset):
    """PyTorch Dataset wrapping preprocessed Europarl token-id sequences."""

    def __init__(self, data_path: str, indices=None, limit: int = None):
        with open(data_path, "rb") as f:
            data = pickle.load(f)
        if indices is not None:
            data = [data[i] for i in indices]
        if limit is not None:
            data = data[:limit]
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.long)


def train_val_indices(n: int, val_fraction: float = Config.val_fraction,
                      seed: int = 0):
    """Deterministic split of the training file into (train, val) indices."""
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n, generator=g).tolist()
    n_val = int(n * val_fraction)
    return sorted(perm[n_val:]), sorted(perm[:n_val])


def load_vocab(data_dir: str = Config.data_dir):
    path = os.path.join(data_dir, "vocab.pkl")
    with open(path, "rb") as f:
        v = pickle.load(f)
    return v["word2idx"], v["idx2word"]


def load_sentences(data_dir: str = Config.data_dir):
    path = os.path.join(data_dir, "sentences.pkl")
    with open(path, "rb") as f:
        return pickle.load(f)


def get_dataloader(split: str = "train", batch_size: int = Config.batch_size,
                   data_dir: str = Config.data_dir, shuffle: bool = None,
                   limit: int = None):
    """split: 'train' | 'val' (held out from the training file) | 'test'."""
    if shuffle is None:
        shuffle = split == "train"
    if split == "test":
        ds = EuroparlDataset(os.path.join(data_dir, "test_data.pkl"), limit=limit)
    else:
        path = os.path.join(data_dir, "train_data.pkl")
        with open(path, "rb") as f:
            n = len(pickle.load(f))
        train_idx, val_idx = train_val_indices(n)
        ds = EuroparlDataset(path, train_idx if split == "train" else val_idx,
                             limit=limit)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=torch.cuda.is_available(),
                      drop_last=(split == "train"))

### `models/__init__.py` — 模型包入口

In [ ]:
%%writefile models/__init__.py
from .transceiver import BaselineDeepSC, ImprovedDeepSC
from .channel import AWGNChannel, RayleighChannel, ChannelModel
from .losses import CompositeLoss
from .llm_refiner import LLMRefiner

### `models/channel.py` — 信道模型

AWGN 和 Rayleigh 信道。v2：Rayleigh 归一化为 E|h|²=1，并返回接收端的有效 SNR。

In [ ]:
%%writefile models/channel.py
"""
Differentiable channel models for end-to-end semantic communication training.

Supported:
  * AWGN  — additive white Gaussian noise
  * Rayleigh  — flat block-fading with perfect CSI equalisation

Both channels are parameterised by SNR (dB) and produce noisy / faded
versions of the transmitted signal while allowing gradient flow.  Each
channel also returns the *effective* per-sample SNR seen by the receiver,
which equals the nominal SNR for AWGN and SNR * |h|^2 for Rayleigh fading.
"""

import math
import torch
import torch.nn as nn


class AWGNChannel(nn.Module):
    """y = x + n,   n ~ N(0, sigma^2 I),   sigma = 1/sqrt(SNR_linear)."""

    def forward(self, x: torch.Tensor, snr_db: torch.Tensor):
        """
        Args:
            x:      (B, L, C) power-normalised transmitted signal
            snr_db: (B,) per-sample SNR in dB
        Returns:
            y:          same shape as x
            snr_eff_db: (B,) effective SNR at the receiver (= snr_db)
        """
        snr_linear = 10.0 ** (snr_db / 10.0)                         # (B,)
        sigma = (1.0 / torch.sqrt(snr_linear)).view(-1, 1, 1)        # (B,1,1)
        noise = sigma * torch.randn_like(x)
        return x + noise, snr_db


class RayleighChannel(nn.Module):
    """
    Block Rayleigh fading:  y = h * x + n
      h = |g|,  g ~ CN(0, 1)  (so E[|h|^2] = 1)  — one coefficient per sample
      n ~ N(0, sigma^2)

    With perfect CSI at receiver:  x_hat = y / h
    """

    def forward(self, x: torch.Tensor, snr_db: torch.Tensor):
        snr_linear = 10.0 ** (snr_db / 10.0)
        sigma = (1.0 / torch.sqrt(snr_linear)).view(-1, 1, 1)

        # Rayleigh fading coefficient (one per sample, constant across time & dim)
        h_real = torch.randn(x.size(0), 1, 1, device=x.device) * math.sqrt(0.5)
        h_imag = torch.randn(x.size(0), 1, 1, device=x.device) * math.sqrt(0.5)
        h = torch.sqrt(h_real ** 2 + h_imag ** 2)                    # (B,1,1)

        noise = sigma * torch.randn_like(x)
        y = h * x + noise

        # Perfect-CSI equalisation
        y_eq = y / (h + 1e-10)
        snr_eff_db = snr_db + 10.0 * torch.log10(h.view(-1) ** 2 + 1e-10)
        return y_eq, snr_eff_db


class ChannelModel(nn.Module):
    """Unified wrapper that selects AWGN or Rayleigh at runtime."""

    def __init__(self):
        super().__init__()
        self.awgn = AWGNChannel()
        self.rayleigh = RayleighChannel()

    def forward(self, x: torch.Tensor, snr_db: torch.Tensor,
                channel_type: str = "awgn"):
        """Returns (y, snr_eff_db)."""
        if channel_type == "rayleigh":
            return self.rayleigh(x, snr_db)
        return self.awgn(x, snr_db)

### `models/modules.py` — 网络模块

SNR 自适应注意力、编码/解码层。v2：信道编解码器加入 FiLM（感知 SNR），以及只给有效 token 分配能量的功率归一化。

In [ ]:
%%writefile models/modules.py
"""
Core neural-network building blocks for the semantic transceiver.

Key innovation:  **SNR-Adaptive Multi-Head Attention**
  -- embeds the current channel SNR into the Transformer's Query vectors,
     allowing the encoder / decoder to dynamically adjust which semantic
     features to prioritise under varying channel conditions.
  Reference: SNR-EQ-JSCC (arXiv 2501.04732, 2025)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ====================================================================
# Positional Encoding (standard sinusoidal)
# ====================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))          # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


# ====================================================================
# SNR Embedding  –  maps scalar SNR (dB) → d_model vector
# ====================================================================
class SNREmbedding(nn.Module):
    """Two-layer MLP that lifts a scalar SNR value to a d_model-dim vector."""

    def __init__(self, d_model: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, snr_db: torch.Tensor):
        """
        Args:
            snr_db: (B,) or (B,1) — SNR in dB.
        Returns:
            (B, d_model)
        """
        if snr_db.dim() == 1:
            snr_db = snr_db.unsqueeze(-1)
        return self.net(snr_db)


# ====================================================================
# SNR-Adaptive Multi-Head Attention
# ====================================================================
class SNRAdaptiveMultiHeadAttention(nn.Module):
    """
    Standard scaled dot-product MHA augmented with a
    Channel-Adaptive Query (CAQ) path.

    Q_adaptive = W_q(x) + W_caq(SNR_emb)

    This lets attention scores shift according to channel quality —
    e.g. under low SNR the model can learn to put more weight on
    high-energy / robust semantic features.
    """

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.caq = nn.Linear(d_model, d_model)       # channel-adaptive query

        self.attn_drop = nn.Dropout(dropout)

    def forward(self, q, k, v, snr_emb, mask=None):
        """
        Args:
            q, k, v: (B, L, d_model)
            snr_emb: (B, d_model) from SNREmbedding
            mask:    (B, 1, 1, L) or broadcastable
        Returns:
            (B, L, d_model)
        """
        B = q.size(0)

        Q = self.W_q(q) + self.caq(snr_emb).unsqueeze(1)   # CAQ injection
        K = self.W_k(k)
        V = self.W_v(v)

        # reshape → (B, n_heads, L, d_k)
        Q = Q.view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(B, -1, self.n_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)
        attn = self.attn_drop(F.softmax(scores, dim=-1))
        ctx = torch.matmul(attn, V)

        ctx = ctx.transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.W_o(ctx)


# ====================================================================
# Standard Multi-Head Attention (no SNR awareness — used in baseline)
# ====================================================================
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.attn_drop = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        Q = self.W_q(q).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)
        attn = self.attn_drop(F.softmax(scores, dim=-1))
        ctx = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.W_o(ctx)


# ====================================================================
# Feed-Forward Network
# ====================================================================
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


# ====================================================================
# SNR-Adaptive Encoder Layer
# ====================================================================
class SNRAdaptiveEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = SNRAdaptiveMultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, snr_emb, mask=None):
        _x = self.norm1(x)
        x = x + self.drop1(self.self_attn(_x, _x, _x, snr_emb, mask))
        _x = self.norm2(x)
        x = x + self.drop2(self.ff(_x))
        return x


# ====================================================================
# Baseline Encoder Layer (no SNR awareness)
# ====================================================================
class BaselineEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        _x = self.norm1(x)
        x = x + self.drop1(self.self_attn(_x, _x, _x, mask))
        _x = self.norm2(x)
        x = x + self.drop2(self.ff(_x))
        return x


# ====================================================================
# Decoder Layer  –  cross-attention to encoder memory + causal self-attn
# ====================================================================
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)
        self.drop3 = nn.Dropout(dropout)

    def forward(self, x, memory, tgt_mask=None, mem_mask=None):
        _x = self.norm1(x)
        x = x + self.drop1(self.self_attn(_x, _x, _x, tgt_mask))
        _x = self.norm2(x)
        x = x + self.drop2(self.cross_attn(_x, memory, memory, mem_mask))
        _x = self.norm3(x)
        x = x + self.drop3(self.ff(_x))
        return x


# ====================================================================
# SNR-Adaptive Decoder Layer  –  cross-attention is channel-aware
# ====================================================================
class SNRAdaptiveDecoderLayer(nn.Module):
    """Decoder layer whose cross-attention uses SNRAdaptiveMultiHeadAttention.

    Self-attention remains standard (the target sequence has no direct
    relationship to channel quality), but cross-attention benefits from
    knowing the SNR so the decoder can weight memory features according
    to channel reliability.
    """

    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = SNRAdaptiveMultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)
        self.drop3 = nn.Dropout(dropout)

    def forward(self, x, memory, snr_emb, tgt_mask=None, mem_mask=None):
        _x = self.norm1(x)
        x = x + self.drop1(self.self_attn(_x, _x, _x, tgt_mask))
        _x = self.norm2(x)
        x = x + self.drop2(self.cross_attn(_x, memory, memory, snr_emb, mem_mask))
        _x = self.norm3(x)
        x = x + self.drop3(self.ff(_x))
        return x


# ====================================================================
# FiLM  –  SNR-conditioned feature-wise affine modulation
# ====================================================================
class FiLM(nn.Module):
    """h -> h * (1 + gamma(c)) + beta(c).

    Zero-initialised so that at the start of training the layer is the
    identity and the model behaves exactly like its un-conditioned version.
    """

    def __init__(self, cond_dim: int, feat_dim: int):
        super().__init__()
        self.proj = nn.Linear(cond_dim, 2 * feat_dim)
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, h, cond):
        """h: (B, L, F),  cond: (B, cond_dim)."""
        gamma, beta = self.proj(cond).unsqueeze(1).chunk(2, dim=-1)
        return h * (1 + gamma) + beta


# ====================================================================
# Channel encoder / decoder  (dense projection + power normalisation)
# ====================================================================
class ChannelEncoder(nn.Module):
    """Maps d_model → channel_dim per time-step, then power-normalises.

    Args:
        film_dim:      if set, the hidden layer is FiLM-modulated by an
                       SNR embedding of this size.
        masked_power:  if True, padded positions transmit zero energy and
                       the per-sentence energy budget (L * channel_dim) is
                       spent on the valid tokens only.  Otherwise every
                       position, padding included, gets unit average power.
    """

    def __init__(self, d_model: int, channel_dim: int,
                 film_dim: int = None, masked_power: bool = False):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Linear(256, channel_dim),
        )
        self.film = FiLM(film_dim, 256) if film_dim else None
        self.masked_power = masked_power

    def forward(self, x, snr_emb=None, valid_mask=None):
        h = self.net[1](self.net[0](x))
        if self.film is not None:
            h = self.film(h, snr_emb)
        out = self.net[2](h).float()                # (B, L, channel_dim)

        if self.masked_power and valid_mask is not None:
            out = out * valid_mask.unsqueeze(-1).to(out.dtype)
            budget = out.size(-2) * out.size(-1)    # same energy as unit power on all slots
            energy = torch.sum(out ** 2, dim=(-2, -1), keepdim=True) + 1e-12
            return out * torch.sqrt(budget / energy)

        # power normalisation: unit average power per sample
        power = torch.mean(out ** 2, dim=(-2, -1), keepdim=True) + 1e-12
        return out / torch.sqrt(power)


class ChannelDecoder(nn.Module):
    """Maps channel_dim → d_model per time-step (optionally SNR-conditioned)."""

    def __init__(self, channel_dim: int, d_model: int, film_dim: int = None):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(channel_dim, 256),
            nn.ReLU(),
            nn.Linear(256, d_model),
        )
        self.film = FiLM(film_dim, 256) if film_dim else None

    def forward(self, x, snr_emb=None):
        h = self.net[1](self.net[0](x))
        if self.film is not None:
            h = self.film(h, snr_emb)
        return self.net[2](h)

### `models/transceiver.py` — 收发机模型

BaselineDeepSC 与 ImprovedDeepSC；共享的贪心解码、批量束搜索、候选词提取。Improved 的每项改进都可以单独关闭，用于消融实验。

In [ ]:
%%writefile models/transceiver.py
"""
End-to-end semantic communication transceivers.

* BaselineDeepSC  — reproduction of the original DeepSC (Xie et al., 2021)
* ImprovedDeepSC  — enhanced with:
      1. SNR-Adaptive Attention in the semantic encoder / decoder
      2. SNR-conditioned (FiLM) channel encoder / decoder
      3. Length-aware power normalisation (no energy spent on padding)
      4. CSI-aware receiver: decoder conditions on the effective SNR
      5. Feature-denoising composite loss (handled externally in losses.py)
      6. LLM fusion refinement at inference time (handled in llm_refiner.py)

Both models share the decoding utilities in ``_DeepSCBase`` (greedy,
batched beam search, per-position candidate extraction).
"""

import torch
import torch.nn as nn

from .modules import (
    PositionalEncoding,
    SNREmbedding,
    SNRAdaptiveEncoderLayer,
    BaselineEncoderLayer,
    DecoderLayer,
    SNRAdaptiveDecoderLayer,
    ChannelEncoder,
    ChannelDecoder,
)
from .channel import ChannelModel


# ====================================================================
# helpers
# ====================================================================
def _causal_mask(seq_len: int, device: torch.device):
    """Lower-triangular causal mask: (1, 1, L, L)."""
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.unsqueeze(0).unsqueeze(0)


def _pad_mask(ids: torch.Tensor, pad_idx: int = 0):
    """(B, 1, 1, L) — True where NOT padded."""
    return (ids != pad_idx).unsqueeze(1).unsqueeze(2)


def _repeat_ctx(ctx, k):
    return None if ctx is None else ctx.repeat_interleave(k, dim=0)


# ====================================================================
# Shared decoding utilities
# ====================================================================
class _DeepSCBase(nn.Module):
    """Subclasses implement ``_transmit`` and ``_decode``.

    _transmit(src, snr_db, channel_type) -> (enc_out, memory, src_mask, ctx)
    _decode(trg, memory, src_mask, ctx)  -> logits (B, L_trg, V)
    """

    def forward(self, src, trg, snr_db, channel_type="awgn"):
        """
        Args:
            src: (B, L_src) source token ids
            trg: (B, L_trg) target token ids (teacher forcing)
            snr_db: (B,) SNR in dB
            channel_type: 'awgn' | 'rayleigh'
        Returns:
            logits: (B, L_trg, V)
            enc_out: (B, L_src, d_model)
            ch_dec_out: (B, L_src, d_model)
        """
        enc_out, memory, src_mask, ctx = self._transmit(src, snr_db, channel_type)
        return self._decode(trg, memory, src_mask, ctx), enc_out, memory

    # ------------------------------------------------------------------
    def _greedy(self, memory, src_mask, ctx, max_len, start_idx, end_idx):
        B = memory.size(0)
        ys = torch.full((B, 1), start_idx, dtype=torch.long, device=memory.device)
        for _ in range(max_len - 1):
            logits = self._decode(ys, memory, src_mask, ctx)[:, -1:, :]
            next_tok = logits.argmax(dim=-1)
            ys = torch.cat([ys, next_tok], dim=1)
            if (next_tok == end_idx).all():
                break
        return ys

    def _beam(self, memory, src_mask, ctx, max_len, start_idx, end_idx, beam_width):
        """Batched beam search: all B*K hypotheses are decoded together."""
        B, K = memory.size(0), beam_width
        device = memory.device
        memory = memory.repeat_interleave(K, dim=0)
        src_mask = src_mask.repeat_interleave(K, dim=0)
        ctx = _repeat_ctx(ctx, K)

        ys = torch.full((B * K, 1), start_idx, dtype=torch.long, device=device)
        scores = torch.full((B, K), float("-inf"), device=device)
        scores[:, 0] = 0.0                      # only one live hypothesis at start
        finished = torch.zeros(B * K, dtype=torch.bool, device=device)
        offset = (torch.arange(B, device=device) * K).unsqueeze(1)
        V = self.vocab_size
        # finished hypotheses can only be extended with PAD, at no cost
        pad_only = torch.full((1, V), float("-inf"), device=device)
        pad_only[0, self.pad_idx] = 0.0

        for _ in range(max_len - 1):
            logits = self._decode(ys, memory, src_mask, ctx)[:, -1, :].float()
            logp = torch.log_softmax(logits, dim=-1)              # (BK, V)
            logp = torch.where(finished.unsqueeze(1), pad_only, logp)

            cand = (scores.view(-1, 1) + logp).view(B, K * V)
            scores, top_idx = cand.topk(K, dim=-1)                # (B, K)
            sel = (offset + top_idx // V).view(-1)
            tok = (top_idx % V).view(-1)

            ys = torch.cat([ys[sel], tok.unsqueeze(1)], dim=1)
            finished = finished[sel] | (tok == end_idx)
            if finished.all():
                break

        best = offset.view(-1) + scores.argmax(dim=-1)
        return ys[best]

    # ------------------------------------------------------------------
    @torch.no_grad()
    def decode(self, src, snr_db, channel_type="awgn", max_len=32,
               start_idx=1, end_idx=2, beam_width=1, topk=0):
        """Transmit ``src`` once and decode it.

        Args:
            beam_width: 1 = greedy, >1 = batched beam search.
            topk:       if >0, also return decoder candidates for every
                        output position (used by the LLM refiner).
        Returns:
            ys:   (B, L) token ids starting with START
            info: None, or dict with
                  conf      (B, L-1) probability of the chosen token ys[:, 1:]
                  cand_ids  (B, L-1, topk) top-k token ids at each position
                  cand_logp (B, L-1, topk) their log-probabilities
        """
        self.eval()
        _, memory, src_mask, ctx = self._transmit(src, snr_db, channel_type)
        if beam_width > 1:
            ys = self._beam(memory, src_mask, ctx, max_len, start_idx, end_idx,
                            beam_width)
        else:
            ys = self._greedy(memory, src_mask, ctx, max_len, start_idx, end_idx)
        if topk <= 0:
            return ys, None

        # one teacher-forced pass over the decoded sequence, same channel output
        logits = self._decode(ys[:, :-1], memory, src_mask, ctx).float()
        logp = torch.log_softmax(logits, dim=-1)
        conf = logp.gather(-1, ys[:, 1:].unsqueeze(-1)).squeeze(-1).exp()
        cand_logp, cand_ids = logp.topk(topk, dim=-1)
        return ys, {"conf": conf, "cand_ids": cand_ids, "cand_logp": cand_logp}

    @torch.no_grad()
    def greedy_decode(self, src, snr_db, channel_type="awgn",
                      max_len=32, start_idx=1, end_idx=2):
        """Auto-regressive greedy decoding for inference."""
        return self.decode(src, snr_db, channel_type, max_len,
                           start_idx, end_idx)[0]

    @torch.no_grad()
    def beam_search_decode(self, src, snr_db, channel_type="awgn",
                           max_len=32, start_idx=1, end_idx=2, beam_width=5):
        return self.decode(src, snr_db, channel_type, max_len,
                           start_idx, end_idx, beam_width=beam_width)[0]


# ====================================================================
# Baseline DeepSC
# ====================================================================
class BaselineDeepSC(_DeepSCBase):
    """Faithful re-implementation of the original DeepSC architecture."""

    def __init__(self, vocab_size, d_model=128, n_heads=8, d_ff=512,
                 n_enc_layers=4, n_dec_layers=4, channel_dim=16,
                 max_len=32, dropout=0.1, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.d_model = d_model
        self.vocab_size = vocab_size

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)

        self.enc_layers = nn.ModuleList([
            BaselineEncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_enc_layers)
        ])
        self.enc_norm = nn.LayerNorm(d_model)
        self.ch_enc = ChannelEncoder(d_model, channel_dim)
        self.channel = ChannelModel()
        self.ch_dec = ChannelDecoder(channel_dim, d_model)

        self.dec_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_dec_layers)
        ])
        self.dec_norm = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def encode(self, src, src_mask=None):
        x = self.pos_enc(self.embedding(src))
        for layer in self.enc_layers:
            x = layer(x, src_mask)
        return self.enc_norm(x)

    def _transmit(self, src, snr_db, channel_type):
        src_mask = _pad_mask(src, self.pad_idx)
        enc_out = self.encode(src, src_mask)
        tx = self.ch_enc(enc_out)
        rx, _ = self.channel(tx, snr_db, channel_type)
        return enc_out, self.ch_dec(rx), src_mask, None

    def _decode(self, trg, memory, src_mask, ctx):
        x = self.pos_enc(self.embedding(trg))
        trg_mask = _causal_mask(trg.size(1), trg.device)
        for layer in self.dec_layers:
            x = layer(x, memory, trg_mask, src_mask)
        return self.fc_out(self.dec_norm(x))


# ====================================================================
# Improved DeepSC
# ====================================================================
class ImprovedDeepSC(_DeepSCBase):
    """
    Enhanced DeepSC.  Each contribution can be switched off for ablation:

      use_film      SNR-conditioned FiLM in the channel encoder / decoder
      masked_power  power normalisation over valid tokens only
      csi_snr       receiver conditions on the effective (post-fading) SNR
                    instead of the average SNR (no effect on AWGN)
    """

    SNR_EFF_RANGE = (-30.0, 40.0)   # clamp for deep fades before embedding

    def __init__(self, vocab_size, d_model=128, n_heads=8, d_ff=512,
                 n_enc_layers=4, n_dec_layers=4, channel_dim=16,
                 max_len=32, dropout=0.1, pad_idx=0,
                 use_film=True, masked_power=True, csi_snr=True):
        super().__init__()
        self.pad_idx = pad_idx
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.use_film = use_film
        self.csi_snr = csi_snr

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)

        # SNR embedding shared across encoder, channel coder and decoder
        self.snr_emb = SNREmbedding(d_model)

        # SNR-adaptive encoder
        self.enc_layers = nn.ModuleList([
            SNRAdaptiveEncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_enc_layers)
        ])
        self.enc_norm = nn.LayerNorm(d_model)

        film_dim = d_model if use_film else None
        self.ch_enc = ChannelEncoder(d_model, channel_dim, film_dim, masked_power)
        self.channel = ChannelModel()
        self.ch_dec = ChannelDecoder(channel_dim, d_model, film_dim)

        # SNR-adaptive decoder (cross-attention is channel-aware)
        self.dec_layers = nn.ModuleList([
            SNRAdaptiveDecoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_dec_layers)
        ])
        self.dec_norm = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def encode(self, src, snr_e, src_mask=None):
        x = self.pos_enc(self.embedding(src))
        for layer in self.enc_layers:
            x = layer(x, snr_e, src_mask)
        return self.enc_norm(x)

    def _transmit(self, src, snr_db, channel_type):
        src_mask = _pad_mask(src, self.pad_idx)
        # transmitter only knows the average SNR
        snr_tx = self.snr_emb(snr_db)
        enc_out = self.encode(src, snr_tx, src_mask)

        tx = self.ch_enc(enc_out, snr_tx, src != self.pad_idx)
        rx, snr_eff = self.channel(tx, snr_db, channel_type)

        # receiver has perfect CSI, so it may condition on the effective SNR
        if self.csi_snr:
            snr_rx = self.snr_emb(snr_eff.clamp(*self.SNR_EFF_RANGE))
        else:
            snr_rx = snr_tx
        return enc_out, self.ch_dec(rx, snr_rx), src_mask, snr_rx

    def _decode(self, trg, memory, src_mask, snr_rx):
        x = self.pos_enc(self.embedding(trg))
        trg_mask = _causal_mask(trg.size(1), trg.device)
        for layer in self.dec_layers:
            x = layer(x, memory, snr_rx, trg_mask, src_mask)
        return self.fc_out(self.dec_norm(x))

### `models/losses.py` — 损失函数

交叉熵 + 逐 token 特征去噪损失（v2 替换了原来的语义损失）。

In [ ]:
%%writefile models/losses.py
"""
Composite loss for semantic communication training.

L_total = alpha * L_CE  +  beta * L_feat

* L_CE    — standard cross-entropy on per-token predictions.
* L_feat  — per-token feature-denoising loss: for every valid position,
            1 - cos(channel-decoder output, stop_grad(semantic-encoder output)).
            It trains the channel decoder to recover the clean semantic
            features of each token from the noisy received signal.  The
            target is detached so the encoder cannot trivially satisfy the
            term by collapsing its features.

For *evaluation* we additionally use external metrics (BLEU, BERTScore)
computed in evaluate.py, but those are not used during training to
keep the training loop efficient and fully differentiable.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class FeatureDenoisingLoss(nn.Module):
    """Mean over valid tokens of 1 - cos(ch_dec_out_i, sg(enc_out_i))."""

    def forward(self, enc_out: torch.Tensor, ch_dec_out: torch.Tensor,
                valid_mask: torch.Tensor):
        """
        Args:
            enc_out:    (B, L, d_model) — semantic encoder output
            ch_dec_out: (B, L, d_model) — channel decoder output
            valid_mask: (B, L) — True where NOT padded
        """
        cos = F.cosine_similarity(ch_dec_out.float(), enc_out.detach().float(),
                                  dim=-1)                              # (B, L)
        m = valid_mask.float()
        return ((1.0 - cos) * m).sum() / m.sum().clamp(min=1)


class CompositeLoss(nn.Module):
    """L = alpha * CE  +  beta * FeatureDenoising."""

    def __init__(self, vocab_size: int, pad_idx: int = 0,
                 alpha: float = 1.0, beta: float = 0.1):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.ce = nn.CrossEntropyLoss(ignore_index=pad_idx)
        self.feat = FeatureDenoisingLoss()

    def forward(self, logits, targets, enc_out=None, ch_dec_out=None,
                pad_mask=None):
        """Returns (total_loss, ce_loss, feat_loss)."""
        ce_loss = self.ce(logits.float().reshape(-1, logits.size(-1)),
                          targets.reshape(-1))

        feat_loss = torch.zeros((), device=logits.device)
        if self.beta > 0 and enc_out is not None and ch_dec_out is not None:
            feat_loss = self.feat(enc_out, ch_dec_out, pad_mask)

        total = self.alpha * ce_loss + self.beta * feat_loss
        return total, ce_loss, feat_loss

### `models/llm_refiner.py` — BERT 融合修正

只在解码器不确定的位置，用 BERT 在解码器自己的 top-5 候选词里重新打分。

In [ ]:
%%writefile models/llm_refiner.py
"""
LLM-Aided Semantic Decoder  (decoder / BERT-MLM fusion refinement)

At inference time the transceiver produces a reconstruction together with,
for every output position, its confidence and its top-k candidate tokens.
The refiner then:

  1. Selects positions whose decoder confidence is below ``tau`` (at most
     ``max_corrections`` per sentence, least confident first, never <UNK>).
  2. Masks those positions and runs a frozen BERT masked-LM once.
  3. Re-scores only the decoder's own top-k candidates at each masked
     position with   log p_dec(w) + lambda * log p_BERT(w | context)
     and keeps the best one.

Restricting the choice to the decoder's candidates keeps the output inside
the transceiver vocabulary and anchored to what was actually received;
BERT only arbitrates between channel-plausible words.  lambda = 0 leaves
the sentence unchanged, so tuning (lambda, tau) on the validation split
can never do worse than no refinement there.

The refiner is applied **only at inference** — it does not participate
in gradient-based training of the transceiver.
"""

from __future__ import annotations

from typing import Dict, List, Sequence, Tuple

import torch

# One sentence to refine: (words, per-word confidence, per-word candidates)
# where candidates is a list of (word, decoder log-prob) pairs.
RefineItem = Tuple[List[str], List[float], List[List[Tuple[str, float]]]]

SPECIAL_WORDS = {"<PAD>", "<START>", "<END>", "<UNK>"}


class LLMRefiner:
    def __init__(
        self,
        model_name: str = "bert-base-uncased",
        lam: float = 0.5,
        tau: float = 0.7,
        max_corrections: int = 3,
        batch_size: int = 64,
        device: torch.device | str = "cpu",
    ):
        from transformers import AutoTokenizer, AutoModelForMaskedLM

        self.device = torch.device(device)
        self.lam = lam
        self.tau = tau
        self.max_corrections = max_corrections
        self.batch_size = batch_size

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForMaskedLM.from_pretrained(model_name).to(self.device)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad = False

        self.mask_token = self.tokenizer.mask_token
        self.mask_id = self.tokenizer.mask_token_id
        self._piece_cache: Dict[str, int] = {}

    # ------------------------------------------------------------------
    def _first_piece(self, word: str) -> int:
        """BERT id of the first word-piece of ``word`` (cached)."""
        if word not in self._piece_cache:
            pieces = self.tokenizer.tokenize(word)
            self._piece_cache[word] = (
                self.tokenizer.convert_tokens_to_ids(pieces[0])
                if pieces else self.tokenizer.unk_token_id)
        return self._piece_cache[word]

    def _positions(self, item: RefineItem, tau: float) -> List[int]:
        words, confs, _ = item
        low = [i for i in range(min(len(words), len(confs)))
               if confs[i] < tau and words[i] not in SPECIAL_WORDS]
        low.sort(key=lambda i: confs[i])
        return sorted(low[: self.max_corrections])

    @torch.no_grad()
    def _bert_logprobs(self, items: Sequence[RefineItem],
                       positions: List[List[int]]):
        """For each item, a {word_pos: log-prob vector over BERT vocab}."""
        out: List[Dict[int, torch.Tensor]] = [{} for _ in items]
        todo = [i for i, pos in enumerate(positions)
                if pos and len(items[i][0]) >= 3]
        for s in range(0, len(todo), self.batch_size):
            chunk = todo[s:s + self.batch_size]
            texts = []
            for i in chunk:
                masked = list(items[i][0])
                for p in positions[i]:
                    masked[p] = self.mask_token
                texts.append(" ".join(masked))
            enc = self.tokenizer(texts, return_tensors="pt", padding=True,
                                 truncation=True, max_length=128).to(self.device)
            logp = torch.log_softmax(self.model(**enc).logits.float(), dim=-1)
            for row, i in enumerate(chunk):
                mask_idx = (enc["input_ids"][row] == self.mask_id).nonzero(
                    as_tuple=True)[0].tolist()
                # each [MASK] is a single word-piece, in word order
                for p, b in zip(positions[i], mask_idx):
                    out[i][p] = logp[row, b].cpu()
        return out

    # ------------------------------------------------------------------
    def refine_grid(self, items: Sequence[RefineItem], lams: Sequence[float],
                    tau: float) -> Dict[float, List[str]]:
        """Refine every item for several lambdas sharing one BERT pass."""
        positions = [self._positions(it, tau) for it in items]
        bert = self._bert_logprobs(items, positions)
        results = {}
        for lam in lams:
            texts = []
            for it, pos_lp in zip(items, bert):
                words = list(it[0])
                if lam > 0:
                    for p, lp_bert in pos_lp.items():
                        best_w, best_s = words[p], float("-inf")
                        for w, lp_dec in it[2][p]:
                            if w in SPECIAL_WORDS:
                                continue
                            score = lp_dec + lam * lp_bert[self._first_piece(w)].item()
                            if score > best_s:
                                best_w, best_s = w, score
                        words[p] = best_w
                texts.append(" ".join(words))
            results[lam] = texts
        return results

    def refine(self, items: Sequence[RefineItem]) -> List[str]:
        return self.refine_grid(items, [self.lam], self.tau)[self.lam]

### `train.py` — 训练脚本

训练一个模型：学习率预热 + 余弦退火、按验证集 −5/0/5 dB 的平均准确率选 best、混合精度、断点续训。

In [ ]:
%%writefile train.py
"""
Training script for Baseline DeepSC and Improved DeepSC.

Usage examples:

  # Train baseline DeepSC on AWGN channel
  python train.py --model baseline --channel awgn --epochs 80

  # Train improved DeepSC on AWGN channel
  python train.py --model improved --channel awgn --epochs 80

  # Ablations of the improved model
  python train.py --model improved --channel awgn --no_film --tag no_film
  python train.py --model improved --channel awgn --no_masked_power --tag no_maskpow
  python train.py --model improved --channel awgn --beta 0.0 --tag no_feat
  python train.py --model improved --channel rayleigh --no_csi_snr --tag no_csi

  # Resume automatically from <ckpt_dir>/last.pt if it exists
  python train.py --model improved --channel awgn --resume auto
"""

import argparse
import json
import math
import os
import time

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from tqdm import tqdm

from config import Config
from dataset import get_dataloader, load_vocab
from models.transceiver import BaselineDeepSC, ImprovedDeepSC
from models.losses import CompositeLoss
from utils import set_seed, AverageMeter, amp_dtype


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--model", choices=["baseline", "improved"], default="improved")
    p.add_argument("--channel", choices=["awgn", "rayleigh"], default="awgn")
    p.add_argument("--epochs", type=int, default=Config.epochs)
    p.add_argument("--batch_size", type=int, default=Config.batch_size)
    p.add_argument("--lr", type=float, default=Config.lr)
    p.add_argument("--alpha", type=float, default=Config.alpha_ce)
    p.add_argument("--beta", type=float, default=None,
                   help="Feature-denoising loss weight "
                        "(default: 0 for baseline, Config.beta_semantic for improved)")
    p.add_argument("--no_film", action="store_true",
                   help="Improved: disable SNR-conditioned channel coder")
    p.add_argument("--no_masked_power", action="store_true",
                   help="Improved: spend power on padding like the baseline")
    p.add_argument("--no_csi_snr", action="store_true",
                   help="Improved: receiver uses average instead of effective SNR")
    p.add_argument("--amp", action="store_true",
                   help="Mixed precision on CUDA (bf16 if supported, else fp16)")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--tag", type=str, default="",
                   help="Extra tag appended to checkpoint dir name")
    p.add_argument("--resume", type=str, default="",
                   help="Checkpoint path to resume from, or 'auto' for <ckpt_dir>/last.pt")
    p.add_argument("--limit_train_batches", type=int, default=0,
                   help="Smoke testing: stop each epoch after N batches")
    p.add_argument("--limit_val_samples", type=int, default=None,
                   help="Smoke testing: use only N validation sentences")
    args = p.parse_args()
    if args.beta is None:
        args.beta = 0.0 if args.model == "baseline" else Config.beta_semantic
    return args


def build_model(model_type, vocab_size, device, use_film=True,
                masked_power=True, csi_snr=True):
    kwargs = dict(
        vocab_size=vocab_size,
        d_model=Config.d_model,
        n_heads=Config.n_heads,
        d_ff=Config.d_ff,
        n_enc_layers=Config.n_encoder_layers,
        n_dec_layers=Config.n_decoder_layers,
        channel_dim=Config.channel_dim,
        max_len=Config.max_seq_len,
        dropout=Config.dropout,
        pad_idx=Config.PAD_IDX,
    )
    if model_type == "baseline":
        return BaselineDeepSC(**kwargs).to(device)
    return ImprovedDeepSC(use_film=use_film, masked_power=masked_power,
                          csi_snr=csi_snr, **kwargs).to(device)


def build_model_from_ckpt(ckpt, vocab_size, device):
    """Rebuild the exact architecture a checkpoint was trained with."""
    a = ckpt.get("args", {})
    model = build_model(a.get("model", "improved"), vocab_size, device,
                        use_film=not a.get("no_film", False),
                        masked_power=not a.get("no_masked_power", False),
                        csi_snr=not a.get("no_csi_snr", False))
    model.load_state_dict(ckpt["model"])
    return model


def make_scheduler(optimizer, total_steps, warmup_steps, lr, lr_min):
    """Linear warm-up followed by cosine decay to lr_min (stepped per batch)."""
    floor = lr_min / lr

    def factor(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

    return LambdaLR(optimizer, factor)


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler,
                    device, channel_type, epoch, total_epochs, dtype,
                    limit_batches=0):
    model.train()
    loss_m = AverageMeter()
    ce_m = AverageMeter()
    feat_m = AverageMeter()
    acc_m = AverageMeter()

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs}", leave=False)
    for i, batch in enumerate(pbar):
        if limit_batches and i >= limit_batches:
            break
        batch = batch.to(device)
        src = batch
        trg_in = batch[:, :-1]
        trg_out = batch[:, 1:]

        # Random SNR per sample
        snr_db = (torch.rand(batch.size(0), device=device)
                  * (Config.snr_train_high - Config.snr_train_low)
                  + Config.snr_train_low)

        with torch.autocast(device_type=device.type, dtype=dtype,
                            enabled=dtype is not None):
            logits, enc_out, ch_dec_out = model(src, trg_in, snr_db, channel_type)
            total, ce, feat = criterion(
                logits, trg_out,
                enc_out=enc_out, ch_dec_out=ch_dec_out,
                pad_mask=(src != Config.PAD_IDX),
            )

        optimizer.zero_grad()
        scaler.scale(total).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), Config.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        # word-level accuracy (ignoring padding)
        preds = logits.argmax(dim=-1)
        mask = trg_out != Config.PAD_IDX
        correct = ((preds == trg_out) & mask).sum().item()
        n_tokens = mask.sum().item()
        acc = correct / max(n_tokens, 1)

        loss_m.update(total.item(), batch.size(0))
        ce_m.update(ce.item(), batch.size(0))
        feat_m.update(feat.item(), batch.size(0))
        acc_m.update(acc, batch.size(0))

        pbar.set_postfix(loss=f"{loss_m.avg:.4f}",
                         ce=f"{ce_m.avg:.4f}",
                         feat=f"{feat_m.avg:.4f}",
                         acc=f"{acc_m.avg:.4f}")

    return loss_m.avg, ce_m.avg, feat_m.avg, acc_m.avg


@torch.no_grad()
def validate(model, loader, device, channel_type, dtype,
             snr_list=Config.val_snr_list):
    """Teacher-forced token accuracy on the validation split at each SNR.

    Uses a fixed noise seed (without disturbing the training RNG) so that
    epochs are compared on identical channel realisations.
    """
    model.eval()
    accs = {}
    devices = [torch.cuda.current_device()] if device.type == "cuda" else []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(1234)
        for snr_val in snr_list:
            correct = total = 0
            for batch in loader:
                batch = batch.to(device)
                trg_in = batch[:, :-1]
                trg_out = batch[:, 1:]
                snr_db = torch.full((batch.size(0),), snr_val, device=device)
                with torch.autocast(device_type=device.type, dtype=dtype,
                                    enabled=dtype is not None):
                    logits, _, _ = model(batch, trg_in, snr_db, channel_type)
                mask = trg_out != Config.PAD_IDX
                correct += ((logits.argmax(-1) == trg_out) & mask).sum().item()
                total += mask.sum().item()
            accs[float(snr_val)] = correct / max(total, 1)
    return sum(accs.values()) / len(accs), accs


def main():
    args = parse_args()
    set_seed(args.seed)
    Config.ensure_dirs()
    device = Config.device
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
    dtype = amp_dtype(device) if args.amp else None
    print(f"Device: {device}  |  AMP: {dtype}")

    # ---- data ----
    word2idx, idx2word = load_vocab()
    vocab_size = len(word2idx)
    print(f"Vocabulary size: {vocab_size}")
    train_loader = get_dataloader("train", args.batch_size)
    val_loader = get_dataloader("val", args.batch_size,
                                limit=args.limit_val_samples)
    print(f"Train: {len(train_loader.dataset)}  |  Val: {len(val_loader.dataset)}")

    # ---- model ----
    model = build_model(args.model, vocab_size, device,
                        use_film=not args.no_film,
                        masked_power=not args.no_masked_power,
                        csi_snr=not args.no_csi_snr)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {args.model}  |  Parameters: {n_params:,}  |  beta={args.beta}")

    # ---- loss & optimiser ----
    criterion = CompositeLoss(vocab_size, Config.PAD_IDX,
                              alpha=args.alpha, beta=args.beta)
    optimizer = AdamW(model.parameters(), lr=args.lr,
                      weight_decay=Config.weight_decay)
    steps_per_epoch = len(train_loader)
    if args.limit_train_batches:
        steps_per_epoch = min(steps_per_epoch, args.limit_train_batches)
    scheduler = make_scheduler(optimizer, args.epochs * steps_per_epoch,
                               Config.warmup_epochs * steps_per_epoch,
                               args.lr, Config.lr_min)
    scaler = torch.amp.GradScaler("cuda", enabled=dtype == torch.float16)

    # ---- checkpoint dir ----
    tag = f"{args.model}_{args.channel}"
    if args.tag:
        tag += f"_{args.tag}"
    ckpt_dir = os.path.join(Config.checkpoint_dir, tag)
    os.makedirs(ckpt_dir, exist_ok=True)

    start_epoch = 1
    best_val_acc = 0.0
    history = []

    resume_path = args.resume
    if resume_path == "auto":
        resume_path = os.path.join(ckpt_dir, "last.pt")
    if resume_path and os.path.isfile(resume_path):
        ckpt = torch.load(resume_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        if "scheduler" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch = ckpt["epoch"] + 1
        best_val_acc = ckpt.get("best_val_acc", 0.0)
        history = ckpt.get("history", [])
        print(f"Resumed from {resume_path}, epoch {start_epoch}")

    # ---- training loop ----
    for epoch in range(start_epoch, args.epochs + 1):
        t0 = time.time()
        train_loss, ce_loss, feat_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler,
            device, args.channel, epoch, args.epochs, dtype,
            args.limit_train_batches)
        val_acc, val_accs = validate(model, val_loader, device, args.channel, dtype)
        elapsed = time.time() - t0

        record = dict(epoch=epoch, train_loss=train_loss, ce_loss=ce_loss,
                      feat_loss=feat_loss, train_acc=train_acc,
                      val_acc=val_acc, val_acc_per_snr=val_accs,
                      lr=scheduler.get_last_lr()[0], time=elapsed)
        history.append(record)

        per_snr = "  ".join(f"{s:+.0f}dB={a:.4f}" for s, a in val_accs.items())
        print(f"[Epoch {epoch:3d}/{args.epochs}] "
              f"train_loss={train_loss:.4f}  ce={ce_loss:.4f}  feat={feat_loss:.4f}  "
              f"train_acc={train_acc:.4f}  val_acc={val_acc:.4f} ({per_snr})  "
              f"lr={scheduler.get_last_lr()[0]:.2e}  time={elapsed:.1f}s")

        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc

        state = dict(
            epoch=epoch,
            model=model.state_dict(),
            optimizer=optimizer.state_dict(),
            scheduler=scheduler.state_dict(),
            best_val_acc=best_val_acc,
            history=history,
            args=vars(args),
        )
        torch.save(state, os.path.join(ckpt_dir, "last.pt"))
        if is_best:
            torch.save(state, os.path.join(ckpt_dir, "best.pt"))
        with open(os.path.join(ckpt_dir, "history.json"), "w") as f:
            json.dump(history, f, indent=2)

    # ---- mark finished (run_all.py skips finished variants) ----
    with open(os.path.join(ckpt_dir, "DONE"), "w") as f:
        f.write(f"{args.epochs}\n")
    print(f"Training complete.  Best val acc = {best_val_acc:.4f}")
    print(f"Checkpoints saved in {ckpt_dir}")


if __name__ == "__main__":
    main()

### `evaluate.py` — 评估脚本

在 7 个 SNR 下计算 WordAcc / BLEU（含 95% 置信区间）/ BERTScore / 句子相似度；--tune_llm 在验证集上调 BERT 融合参数。

In [ ]:
%%writefile evaluate.py
"""
Comprehensive evaluation of semantic communication models.

Metrics computed per SNR point:
  - Word Accuracy   (token-level match)
  - BLEU-1 … BLEU-4 (n-gram precision; BLEU-4 with a bootstrap 95% CI)
  - BERTScore       (contextual embedding similarity)
  - Sentence Similarity (cosine sim of DistilBERT [CLS] embeddings)

The architecture is rebuilt from the arguments stored in the checkpoint.
The channel noise is re-seeded at every SNR point, so different models
(and greedy vs. beam decoding) see the same noise realisations.

Usage:
  python evaluate.py --checkpoint checkpoints/improved_awgn/best.pt

  python evaluate.py --checkpoint checkpoints/improved_awgn/best.pt \\
                     --beam_width 5 --eval_tag beam5

  # tune the LLM fusion weights on the validation split, then apply them
  python evaluate.py --checkpoint checkpoints/improved_awgn/best.pt \\
                     --beam_width 5 --tune_llm
  python evaluate.py --checkpoint checkpoints/improved_awgn/best.pt \\
                     --beam_width 5 --use_llm --eval_tag beam5_llm
"""

import argparse
import json
import math
import os

import torch
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

from config import Config
from dataset import get_dataloader, load_vocab
from train import build_model_from_ckpt
from utils import set_seed, batch_ids_to_texts


# ====================================================================
# Metric helpers
# ====================================================================
def compute_word_accuracy(pred_ids, ref_ids, pad_idx=0, end_idx=2):
    """Token-level accuracy ignoring PAD.  Returns (correct, total)."""
    correct = 0
    total = 0
    for p, r in zip(pred_ids.tolist(), ref_ids.tolist()):
        for pi, ri in zip(p, r):
            if ri == pad_idx:
                break
            total += 1
            if pi == ri:
                correct += 1
    return correct, total


def compute_bleu(pred_texts, ref_texts, orders=(1, 2, 3, 4)):
    """Average sentence-level BLEU-n, plus the per-sentence BLEU-4 scores."""
    sf = SmoothingFunction().method1
    bleu_scores = {f"BLEU-{n}": [] for n in orders}
    for pred, ref in zip(pred_texts, ref_texts):
        ref_tokens = ref.split()
        pred_tokens = pred.split()
        if len(pred_tokens) == 0:
            pred_tokens = [""]
        for n in orders:
            w = [1.0 / n] * n + [0.0] * (4 - n)
            score = sentence_bleu([ref_tokens], pred_tokens,
                                  weights=w, smoothing_function=sf)
            bleu_scores[f"BLEU-{n}"].append(score)
    means = {k: float(np.mean(v)) for k, v in bleu_scores.items()}
    return means, bleu_scores.get("BLEU-4")


def bootstrap_ci(values, n_boot=1000, seed=0):
    """95% percentile bootstrap CI of the mean."""
    v = np.asarray(values, dtype=np.float64)
    rng = np.random.default_rng(seed)
    means = v[rng.integers(0, len(v), size=(n_boot, len(v)))].mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def bleu_with_ci(pred_texts, ref_texts):
    means, b4 = compute_bleu(pred_texts, ref_texts)
    lo, hi = bootstrap_ci(b4)
    return {**means, "BLEU-4_CI_low": lo, "BLEU-4_CI_high": hi}


_SCORERS = {}


def compute_bert_score(pred_texts, ref_texts, device="cpu"):
    """BERTScore (Precision, Recall, F1) using the bert-score library."""
    try:
        if "bertscore" not in _SCORERS:
            from bert_score import BERTScorer
            _SCORERS["bertscore"] = BERTScorer(lang="en", device=str(device), batch_size=128)
        P, R, F1 = _SCORERS["bertscore"].score(pred_texts, ref_texts, verbose=False)
        return {
            "BERTScore_P": float(P.mean()),
            "BERTScore_R": float(R.mean()),
            "BERTScore_F1": float(F1.mean()),
        }
    except Exception as e:
        print(f"  [warn] BERTScore failed: {e}")
        return {}


def compute_sentence_similarity(pred_texts, ref_texts, device="cpu"):
    """Mean cosine similarity of sentence-level DistilBERT [CLS] embeddings."""
    try:
        if "distilbert" not in _SCORERS:
            from transformers import AutoTokenizer, AutoModel
            tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
            mdl = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
            mdl.eval()
            _SCORERS["distilbert"] = (tok, mdl)
        tok, mdl = _SCORERS["distilbert"]
        sims = []
        bs = 128
        for i in range(0, len(pred_texts), bs):
            p_batch = pred_texts[i:i + bs]
            r_batch = ref_texts[i:i + bs]
            with torch.no_grad():
                pe = tok(p_batch, padding=True, truncation=True,
                         max_length=64, return_tensors="pt").to(device)
                re_ = tok(r_batch, padding=True, truncation=True,
                          max_length=64, return_tensors="pt").to(device)
                p_emb = mdl(**pe).last_hidden_state[:, 0]
                r_emb = mdl(**re_).last_hidden_state[:, 0]
            cos = torch.nn.functional.cosine_similarity(p_emb, r_emb)
            sims.extend(cos.cpu().tolist())
        return {"SentSim": float(np.mean(sims))}
    except Exception as e:
        print(f"  [warn] SentSim failed: {e}")
        return {}


def text_metrics(pred_texts, ref_texts, device, with_embeddings=True):
    m = bleu_with_ci(pred_texts, ref_texts)
    if with_embeddings:
        m.update(compute_bert_score(pred_texts, ref_texts, device))
        m.update(compute_sentence_similarity(pred_texts, ref_texts, device))
    return m


# ====================================================================
# Decoding
# ====================================================================
def snr_seed(snr_val):
    """Same seed for the same SNR, whatever the model or decoding strategy."""
    return 42 + int(round((snr_val + 100) * 10))


def build_refine_items(ys, info, idx2word, end_idx=Config.END_IDX):
    """Per-sentence (words, confidences, candidates) for the LLM refiner.

    Words follow exactly the same rules as ``utils.ids_to_text`` so that
    " ".join(words) equals the unrefined decoded text.
    """
    ys = ys.tolist()
    conf = info["conf"].tolist()
    cand_ids = info["cand_ids"].tolist()
    cand_logp = info["cand_logp"].tolist()
    items = []
    for i, seq in enumerate(ys):
        words, confs, cands = [], [], []
        for j in range(1, len(seq)):
            if seq[j] == end_idx:
                break
            w = idx2word.get(seq[j], "")
            if not w or w in ("<PAD>", "<START>"):
                continue
            c = conf[i][j - 1]
            cs = [(idx2word.get(t, ""), lp)
                  for t, lp in zip(cand_ids[i][j - 1], cand_logp[i][j - 1])]
            if w not in {cw for cw, _ in cs}:
                cs.append((w, math.log(max(c, 1e-12))))
            words.append(w)
            confs.append(c)
            cands.append(cs)
        items.append((words, confs, cands))
    return items


@torch.no_grad()
def decode_at_snr(model, loader, idx2word, snr_val, channel_type, device,
                  max_samples, beam_width=1, topk=0):
    """Decode the split at a fixed SNR.

    Returns pred_texts, ref_texts, (correct, total) word counts and,
    if topk > 0, the refiner items.
    """
    set_seed(snr_seed(snr_val))
    model.eval()
    pred_texts, ref_texts, items = [], [], []
    correct = total = n = 0

    for batch in loader:
        batch = batch[: max_samples - n].to(device)
        B = batch.size(0)
        snr_db = torch.full((B,), snr_val, device=device, dtype=torch.float)
        ys, info = model.decode(batch, snr_db, channel_type,
                                max_len=Config.max_seq_len,
                                start_idx=Config.START_IDX,
                                end_idx=Config.END_IDX,
                                beam_width=beam_width, topk=topk)

        pred = torch.full((B, Config.max_seq_len), Config.PAD_IDX,
                          dtype=torch.long, device=ys.device)
        pred[:, :ys.size(1)] = ys[:, :Config.max_seq_len]
        c, t = compute_word_accuracy(pred.cpu(), batch.cpu(),
                                     Config.PAD_IDX, Config.END_IDX)
        correct += c
        total += t

        pred_texts += batch_ids_to_texts(pred.cpu(), idx2word, Config.END_IDX)
        ref_texts += batch_ids_to_texts(batch.cpu(), idx2word, Config.END_IDX)
        if topk > 0:
            items += build_refine_items(ys.cpu(), {k: v.cpu() for k, v in info.items()},
                                        idx2word)
        n += B
        if n >= max_samples:
            break
    return pred_texts, ref_texts, (correct, total), items


# ====================================================================
# Entry points
# ====================================================================
def load_model(args, device):
    word2idx, idx2word = load_vocab()
    ckpt = torch.load(args.checkpoint, map_location=device, weights_only=False)
    model = build_model_from_ckpt(ckpt, len(word2idx), device)
    model.eval()
    ckpt_args = ckpt.get("args", {})
    channel = args.channel or ckpt_args.get("channel", "awgn")
    print(f"Loaded {ckpt_args.get('model', '?')} from {args.checkpoint} "
          f"(epoch {ckpt.get('epoch')}, channel={channel})")
    return model, idx2word, channel


def default_llm_model():
    has_weights = any(os.path.isfile(os.path.join(Config.bert_dir, f))
                      for f in ("model.safetensors", "pytorch_model.bin"))
    return Config.bert_dir if has_weights else "bert-base-uncased"


def fusion_path(args):
    return os.path.join(os.path.dirname(args.checkpoint),
                        f"llm_fusion_beam{args.beam_width}.json")


def make_refiner(args, device):
    from models.llm_refiner import LLMRefiner
    llm_model = args.llm_model or default_llm_model()
    refiner = LLMRefiner(model_name=llm_model, device=device)
    print(f"LLM refiner loaded ({llm_model})")
    return refiner


def run_tuning(args):
    """Grid-search (lambda, tau) on the validation split; save the best."""
    device = Config.device
    model, idx2word, channel = load_model(args, device)
    refiner = make_refiner(args, device)
    loader = get_dataloader("val", batch_size=args.batch_size, shuffle=False)
    max_samples = args.max_samples or Config.llm_tune_samples
    snrs = args.snr_list or Config.llm_tune_snrs

    grid = {}   # "lam,tau" -> list of BLEU-4 per SNR
    for snr_val in snrs:
        _, refs, _, items = decode_at_snr(model, loader, idx2word, snr_val,
                                          channel, device, max_samples,
                                          args.beam_width, Config.llm_topk)
        for tau in Config.llm_tau_grid:
            outs = refiner.refine_grid(items, Config.llm_lambda_grid, tau)
            for lam, texts in outs.items():
                b4 = compute_bleu(texts, refs, orders=(4,))[0]["BLEU-4"]
                grid.setdefault(f"{lam},{tau}", []).append(b4)
        print(f"  tuned at {snr_val} dB")

    mean = {k: float(np.mean(v)) for k, v in grid.items()}
    # best mean BLEU-4; ties go to the smaller lambda (lambda=0 is no-op)
    best = max(mean, key=lambda k: (round(mean[k], 6), -float(k.split(",")[0])))
    lam, tau = (float(x) for x in best.split(","))
    out = {"lam": lam, "tau": tau, "val_bleu4": mean[best],
           "val_bleu4_no_refine": mean[f"0.0,{Config.llm_tau_grid[0]}"],
           "grid": mean, "snrs": snrs, "beam_width": args.beam_width}
    with open(fusion_path(args), "w") as f:
        json.dump(out, f, indent=2)
    print(f"Best lambda={lam} tau={tau}  val BLEU-4 {mean[best]:.4f} "
          f"(no refinement {out['val_bleu4_no_refine']:.4f})")
    print(f"Saved to {fusion_path(args)}")


def run_evaluation(args):
    device = Config.device
    model, idx2word, channel = load_model(args, device)
    loader = get_dataloader("test", batch_size=args.batch_size, shuffle=False)

    if args.beam_width > 1:
        print(f"Beam search enabled (width={args.beam_width})")

    # optional LLM refiner
    refiner = None
    if args.use_llm:
        refiner = make_refiner(args, device)
        if os.path.isfile(fusion_path(args)):
            with open(fusion_path(args)) as f:
                tuned = json.load(f)
            refiner.lam, refiner.tau = tuned["lam"], tuned["tau"]
            print(f"Using tuned fusion weights lambda={refiner.lam} tau={refiner.tau}")
        else:
            print(f"[warn] {fusion_path(args)} not found; using defaults "
                  f"lambda={refiner.lam} tau={refiner.tau}")

    results = {}
    snr_list = args.snr_list or Config.eval_snr_list
    with_emb = not args.no_bertscore

    for snr_val in snr_list:
        print(f"\n===== SNR = {snr_val} dB  |  channel = {channel} =====")
        pred_texts, ref_texts, (c, t), items = decode_at_snr(
            model, loader, idx2word, snr_val, channel, device,
            args.max_samples, args.beam_width,
            Config.llm_topk if refiner is not None else 0)

        metrics = {"WordAcc": c / max(t, 1),
                   **text_metrics(pred_texts, ref_texts, device, with_emb)}

        for j in range(min(3, len(pred_texts))):
            print(f"  REF : {ref_texts[j]}")
            print(f"  PRED: {pred_texts[j]}")
            print()

        if refiner is not None:
            refined = refiner.refine(items)
            n_changed = sum(a != b for a, b in zip(pred_texts, refined))
            r = text_metrics(refined, ref_texts, device, with_emb)
            metrics.update({f"LLM_{k}": v for k, v in r.items()})
            metrics["LLM_changed_frac"] = n_changed / max(len(refined), 1)
            for a, b, ref in zip(pred_texts, refined, ref_texts):
                if a != b:
                    print(f"  REF : {ref}\n  PRED: {a}\n  LLM : {b}\n")
                    break

        results[int(snr_val)] = metrics
        print("  Metrics:", {k: f"{v:.4f}" for k, v in metrics.items()})

    # ---- save results ----
    out_dir = os.path.dirname(args.checkpoint)
    suffix = f"_{args.eval_tag}" if args.eval_tag else ""
    out_path = os.path.join(out_dir, f"eval_{channel}{suffix}.json")
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {out_path}")
    return results


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--checkpoint", type=str, required=True)
    p.add_argument("--model", choices=["baseline", "improved"], default=None,
                   help="Ignored: the architecture is read from the checkpoint")
    p.add_argument("--channel", choices=["awgn", "rayleigh"], default=None,
                   help="Default: the channel the checkpoint was trained on")
    p.add_argument("--use_llm", action="store_true",
                   help="Apply LLM fusion refinement and report before/after metrics")
    p.add_argument("--tune_llm", action="store_true",
                   help="Tune LLM fusion (lambda, tau) on the validation split")
    p.add_argument("--llm_model", type=str, default=None,
                   help="HuggingFace model name or local path for the refiner "
                        "(default: fine-tuned Europarl BERT if present)")
    p.add_argument("--beam_width", type=int, default=1,
                   help="Beam width for decoding (1=greedy, >1=beam search)")
    p.add_argument("--eval_tag", type=str, default="",
                   help="Tag appended to output filename, e.g. 'beam5'")
    p.add_argument("--max_samples", type=int, default=None)
    p.add_argument("--batch_size", type=int, default=256)
    p.add_argument("--snr_list", type=float, nargs="+", default=None)
    p.add_argument("--no_bertscore", action="store_true",
                   help="Skip BERTScore / SentSim (fast smoke runs)")
    args = p.parse_args()
    if args.tune_llm:
        run_tuning(args)
        return
    if args.max_samples is None:
        args.max_samples = Config.eval_max_samples
    run_evaluation(args)


if __name__ == "__main__":
    main()

### `finetune_bert.py` — BERT 微调

用 Europarl 训练句微调 BERT，供融合修正使用。

In [ ]:
%%writefile finetune_bert.py
"""
Fine-tune bert-base-uncased on Europarl data for Masked Language Modeling.

This produces a domain-adapted BERT MLM that the LLMRefiner can use
to correct low-confidence decoder outputs more accurately.

Usage:
    python finetune_bert.py                        # defaults
    python finetune_bert.py --epochs 5 --lr 2e-5   # custom
"""

import argparse
import os
import pickle
import math

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
)

from config import Config
from dataset import train_val_indices


class EuroparlTextDataset(Dataset):
    """Simple dataset returning raw sentences for BERT tokenizer."""

    def __init__(self, sentences, tokenizer, max_length=128):
        self.encodings = tokenizer(
            sentences,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt",
        )

    def __len__(self):
        return self.encodings["input_ids"].size(0)

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", default="bert-base-uncased")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=2e-5)
    parser.add_argument("--mlm_prob", type=float, default=0.15)
    parser.add_argument("--max_sentences", type=int, default=100_000,
                        help="Max training sentences (for speed)")
    parser.add_argument("--save_dir", default=Config.bert_dir)
    args = parser.parse_args()

    device = Config.device
    os.makedirs(args.save_dir, exist_ok=True)

    sent_path = os.path.join(Config.data_dir, "sentences.pkl")
    with open(sent_path, "rb") as f:
        all_sents = pickle.load(f)
    # exclude the validation sentences used to tune the LLM fusion weights
    train_idx, _ = train_val_indices(len(all_sents["train"]))
    train_sents = [all_sents["train"][i] for i in train_idx[:args.max_sentences]]
    print(f"Fine-tuning on {len(train_sents)} Europarl sentences")

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    model = AutoModelForMaskedLM.from_pretrained(args.model_name).to(device)

    dataset = EuroparlTextDataset(train_sents, tokenizer, max_length=64)
    collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=args.mlm_prob,
    )
    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True,
                        collate_fn=collator, num_workers=0,
                        pin_memory=torch.cuda.is_available())

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.01)
    total_steps = len(loader) * args.epochs
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

    print(f"Device: {device} | Epochs: {args.epochs} | Steps/epoch: {len(loader)}")

    for epoch in range(1, args.epochs + 1):
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(loader, 1):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            if step % 200 == 0:
                avg = total_loss / step
                ppl = math.exp(min(avg, 10))
                print(f"  Epoch {epoch} step {step}/{len(loader)} | "
                      f"loss={avg:.4f} ppl={ppl:.2f}")

        avg_loss = total_loss / len(loader)
        ppl = math.exp(min(avg_loss, 10))
        print(f"Epoch {epoch}/{args.epochs} done | loss={avg_loss:.4f} ppl={ppl:.2f}")

    model.save_pretrained(args.save_dir)
    tokenizer.save_pretrained(args.save_dir)
    print(f"\nFine-tuned model saved to {args.save_dir}")


if __name__ == "__main__":
    main()

### `ablation.py` — 结果汇总

读取所有评估结果，生成 summary.md 和曲线图。

In [ ]:
%%writefile ablation.py
"""
Ablation study result collection and visualization.

Reads the evaluation JSON files written by evaluate.py and produces
comparison plots and a summary table suitable for the final report.

Configurations (see run_all.py):
  AWGN:      Baseline, Improved (full), -FiLM, -masked power, -feature loss,
             Baseline/Improved with beam search, Improved + beam + LLM fusion
  Rayleigh:  Baseline, Improved (full), -CSI-aware SNR, beam / LLM variants

Usage:
  # After training and evaluating all variants, collect & plot
  python ablation.py --results_dir checkpoints --out_dir results
"""

import argparse
import json
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# label -> (checkpoint dir, eval file, metric prefix)
CONFIGS = {
    "awgn": {
        "Baseline":                 ("baseline_awgn", "eval_awgn.json", ""),
        "Improved (full)":          ("improved_awgn", "eval_awgn.json", ""),
        "Improved - FiLM":          ("improved_awgn_no_film", "eval_awgn.json", ""),
        "Improved - masked power":  ("improved_awgn_no_maskpow", "eval_awgn.json", ""),
        "Improved - feat. loss":    ("improved_awgn_no_feat", "eval_awgn.json", ""),
        "Baseline + beam5":         ("baseline_awgn", "eval_awgn_beam5.json", ""),
        "Improved + beam5":         ("improved_awgn", "eval_awgn_beam5.json", ""),
        "Improved + beam5 + LLM":   ("improved_awgn", "eval_awgn_beam5_llm.json", "LLM_"),
    },
    "rayleigh": {
        "Baseline":                 ("baseline_rayleigh", "eval_rayleigh.json", ""),
        "Improved (full)":          ("improved_rayleigh", "eval_rayleigh.json", ""),
        "Improved - CSI SNR":       ("improved_rayleigh_no_csi", "eval_rayleigh.json", ""),
        "Baseline + beam5":         ("baseline_rayleigh", "eval_rayleigh_beam5.json", ""),
        "Improved + beam5":         ("improved_rayleigh", "eval_rayleigh_beam5.json", ""),
        "Improved + beam5 + LLM":   ("improved_rayleigh", "eval_rayleigh_beam5_llm.json", "LLM_"),
    },
}

METRICS = ["WordAcc", "BLEU-1", "BLEU-4", "BERTScore_F1", "SentSim"]
METRIC_LABELS = {
    "WordAcc":       "Word Accuracy",
    "BLEU-1":        "BLEU-1",
    "BLEU-4":        "BLEU-4",
    "BERTScore_F1":  "BERTScore F1",
    "SentSim":       "Sentence Similarity",
}


def load_results(results_dir, channel):
    """{label: {snr: {metric: value}}} for every evaluation file found."""
    data = {}
    for label, (dirname, fname, prefix) in CONFIGS[channel].items():
        fpath = os.path.join(results_dir, dirname, fname)
        if not os.path.isfile(fpath):
            continue
        with open(fpath) as f:
            raw = json.load(f)
        data[label] = {
            int(float(snr)): {k[len(prefix):]: v for k, v in m.items()
                              if k.startswith(prefix) and (prefix or not k.startswith("LLM_"))}
            for snr, m in raw.items()
        }
    return data


def plot_metric_vs_snr(data, metric, channel, out_dir):
    """Plot one metric across SNR for all available configurations."""
    plt.figure(figsize=(8, 5))
    for label, snr_dict in data.items():
        snrs = sorted(snr_dict.keys())
        if not all(metric in snr_dict[s] for s in snrs):
            continue
        vals = [snr_dict[s][metric] for s in snrs]
        style = "--" if "beam" in label else "-"
        plt.plot(snrs, vals, style, marker="o", linewidth=2, label=label)
        if metric == "BLEU-4" and "BLEU-4_CI_low" in snr_dict[snrs[0]]:
            plt.fill_between(snrs,
                             [snr_dict[s]["BLEU-4_CI_low"] for s in snrs],
                             [snr_dict[s]["BLEU-4_CI_high"] for s in snrs],
                             alpha=0.15)
    plt.xlabel("SNR (dB)", fontsize=13)
    plt.ylabel(METRIC_LABELS.get(metric, metric), fontsize=13)
    plt.title(f"{METRIC_LABELS.get(metric, metric)} vs SNR ({channel.upper()})",
              fontsize=14)
    plt.legend(fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    fname = os.path.join(out_dir, f"{channel}_{metric}_vs_snr.png")
    plt.savefig(fname, dpi=150)
    plt.close()
    print(f"  Saved {fname}")


def plot_training_curves(results_dir, out_dir):
    """Plot training loss and validation accuracy from history.json files."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    dirs = sorted({d for cfg in CONFIGS.values() for d, _, _ in cfg.values()})
    for dirname in dirs:
        hpath = os.path.join(results_dir, dirname, "history.json")
        if not os.path.isfile(hpath):
            continue
        with open(hpath) as f:
            hist = json.load(f)
        epochs = [r["epoch"] for r in hist]
        axes[0].plot(epochs, [r["train_loss"] for r in hist], label=dirname)
        axes[1].plot(epochs, [r["val_acc"] for r in hist], label=dirname)
    axes[0].set(xlabel="Epoch", ylabel="Training Loss",
                title="Training Loss Curves")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)
    axes[1].set(xlabel="Epoch", ylabel="Validation Accuracy (mean of -5/0/5 dB)",
                title="Validation Accuracy Curves")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    fname = os.path.join(out_dir, "training_curves.png")
    plt.savefig(fname, dpi=150)
    plt.close()
    print(f"  Saved {fname}")


def summary_table(data, channel, snrs=(-10, -5, 0, 5, 10)):
    """Markdown table of BLEU-4 (with 95% CI) and WordAcc."""
    lines = [f"### {channel.upper()}", ""]
    for metric in ["BLEU-4", "WordAcc", "BERTScore_F1"]:
        lines.append(f"**{METRIC_LABELS[metric]}**\n")
        lines.append("| Configuration | " + " | ".join(f"{s} dB" for s in snrs) + " |")
        lines.append("|---|" + "---|" * len(snrs))
        for label, snr_dict in data.items():
            cells = []
            for s in snrs:
                m = snr_dict.get(s, {})
                if metric not in m:
                    cells.append("–")
                elif metric == "BLEU-4" and "BLEU-4_CI_low" in m:
                    half = (m["BLEU-4_CI_high"] - m["BLEU-4_CI_low"]) / 2
                    cells.append(f"{m[metric]:.4f} ± {half:.4f}")
                else:
                    cells.append(f"{m[metric]:.4f}")
            lines.append(f"| {label} | " + " | ".join(cells) + " |")
        lines.append("")
    return "\n".join(lines)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--results_dir", type=str, default="checkpoints")
    p.add_argument("--out_dir", type=str, default="results")
    args = p.parse_args()
    os.makedirs(args.out_dir, exist_ok=True)

    tables = []
    for channel in CONFIGS:
        data = load_results(args.results_dir, channel)
        if not data:
            print(f"No {channel} evaluation files found.")
            continue
        print(f"Loaded {channel} results for: {list(data.keys())}")
        for metric in METRICS:
            plot_metric_vs_snr(data, metric, channel, args.out_dir)
        tables.append(summary_table(data, channel))

    plot_training_curves(args.results_dir, args.out_dir)

    if tables:
        text = "\n\n".join(tables)
        with open(os.path.join(args.out_dir, "summary.md"), "w") as f:
            f.write(text + "\n")
        print("\n" + text)

    print(f"\nAll plots saved to {args.out_dir}/")


if __name__ == "__main__":
    main()

### `run_all.py` — 总调度

依次训练 8 个模型 → 评估 → 微调 BERT → 调参 → 出图。可重复运行：已完成的步骤会跳过。

In [ ]:
%%writefile run_all.py
"""
Convenience script that trains all model variants sequentially, then
fine-tunes the refiner BERT, evaluates everything and plots the ablation.

The script is idempotent: finished variants (DONE marker) are skipped,
interrupted ones resume from last.pt, and existing evaluation files are
not recomputed.  Re-running the same command after a disconnect continues
where it stopped.

Usage:
  python run_all.py --amp               # full pipeline on a CUDA GPU
  python run_all.py --epochs 20         # shorter run
  python run_all.py --skip_train        # only evaluate + plot
  python run_all.py --smoke             # tiny end-to-end check (CPU ok)
"""

import argparse
import os
import subprocess
import sys

from config import Config


TRAIN_CONFIGS = [
    # (dir name, model, channel, extra train args)
    ("baseline_awgn",            "baseline", "awgn",     []),
    ("improved_awgn",            "improved", "awgn",     []),
    ("improved_awgn_no_film",    "improved", "awgn",     ["--no_film", "--tag", "no_film"]),
    ("improved_awgn_no_maskpow", "improved", "awgn",     ["--no_masked_power", "--tag", "no_maskpow"]),
    ("improved_awgn_no_feat",    "improved", "awgn",     ["--beta", "0.0", "--tag", "no_feat"]),
    ("baseline_rayleigh",        "baseline", "rayleigh", []),
    ("improved_rayleigh",        "improved", "rayleigh", []),
    ("improved_rayleigh_no_csi", "improved", "rayleigh", ["--no_csi_snr", "--tag", "no_csi"]),
]

# Main models get the full metric set (BERTScore, SentSim) and beam search.
# Ablation variants only need BLEU / WordAcc with greedy decoding to show
# each component's contribution, which keeps evaluation time manageable.
MAIN_CONFIGS = ["baseline_awgn", "improved_awgn", "baseline_rayleigh", "improved_rayleigh"]
# models that additionally get the LLM fusion refiner
LLM_CONFIGS = ["improved_awgn", "improved_rayleigh"]
BEAM = 5


ROOT = os.path.dirname(os.path.abspath(__file__))


def run_cmd(cmd, env):
    print(f"\n{'='*60}")
    print(f"  Running: {' '.join(cmd)}")
    print(f"{'='*60}\n")
    subprocess.run(cmd, check=True, env=env, cwd=ROOT)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--epochs", type=int, default=Config.epochs)
    p.add_argument("--amp", action="store_true", help="Mixed precision training")
    p.add_argument("--skip_train", action="store_true")
    p.add_argument("--skip_eval", action="store_true")
    p.add_argument("--no_llm", action="store_true", help="Skip BERT fine-tuning and refinement")
    p.add_argument("--smoke", action="store_true",
                   help="Tiny run into smoke_checkpoints/ to check the pipeline")
    args = p.parse_args()

    py = sys.executable
    env = dict(os.environ)
    train_extra, eval_extra, bert_extra = [], [], []
    if args.smoke:
        # always isolated, so a smoke run can never mark real runs as DONE
        env["SC_CKPT_DIR"] = os.path.join(ROOT, "smoke_checkpoints")
        env["SC_RESULT_DIR"] = os.path.join(ROOT, "smoke_checkpoints", "results")
        args.epochs = 1
        train_extra = ["--limit_train_batches", "20", "--limit_val_samples", "256"]
        eval_extra = ["--max_samples", "64", "--snr_list", "0", "--no_bertscore"]
        bert_extra = ["--epochs", "1", "--max_sentences", "64"]
    if args.amp:
        train_extra.append("--amp")
    ckpt_root = env.get("SC_CKPT_DIR", Config.checkpoint_dir)
    result_dir = env.get("SC_RESULT_DIR", Config.result_dir)

    def ckpt_of(tag):
        path = os.path.join(ckpt_root, tag, "best.pt")
        return path if os.path.isfile(path) else os.path.join(ckpt_root, tag, "last.pt")

    def evaluate(tag, channel, extra, eval_tag):
        out = os.path.join(ckpt_root, tag, f"eval_{channel}{'_' + eval_tag if eval_tag else ''}.json")
        if os.path.isfile(out) and not args.smoke:
            print(f"  {out} exists, skipping.")
            return
        cmd = [py, "evaluate.py", "--checkpoint", ckpt_of(tag)] + extra
        if eval_tag:
            cmd += ["--eval_tag", eval_tag]
        run_cmd(cmd + eval_extra, env)

    # ---- Training ----
    if not args.skip_train:
        for tag, model, channel, extras in TRAIN_CONFIGS:
            if os.path.isfile(os.path.join(ckpt_root, tag, "DONE")) and not args.smoke:
                print(f"  {tag} already trained, skipping.")
                continue
            cmd = [py, "train.py",
                   "--model", model,
                   "--channel", channel,
                   "--epochs", str(args.epochs),
                   "--resume", "auto"] + extras + train_extra
            run_cmd(cmd, env)

    # ---- Evaluation ----
    if not args.skip_eval:
        for tag, _, channel, _ in TRAIN_CONFIGS:
            if not os.path.isfile(ckpt_of(tag)):
                print(f"  Checkpoint not found for {tag}, skipping eval.")
                continue
            if tag in MAIN_CONFIGS:
                evaluate(tag, channel, [], "")
                # beam vs greedy is compared on BLEU; BERTScore is the slowest metric
                evaluate(tag, channel, ["--beam_width", str(BEAM), "--no_bertscore"], f"beam{BEAM}")
            else:
                evaluate(tag, channel, ["--no_bertscore"], "")

        if not args.no_llm:
            bert_dir = os.path.join(ckpt_root, "bert_europarl")
            if not os.path.isfile(os.path.join(bert_dir, "config.json")) or args.smoke:
                run_cmd([py, "finetune_bert.py", "--save_dir", bert_dir] + bert_extra, env)
            for tag in LLM_CONFIGS:
                channel = "rayleigh" if "rayleigh" in tag else "awgn"
                if not os.path.isfile(ckpt_of(tag)):
                    continue
                beam = ["--beam_width", str(BEAM)]
                fusion = os.path.join(ckpt_root, tag, f"llm_fusion_beam{BEAM}.json")
                if not os.path.isfile(fusion) or args.smoke:
                    tune_extra = ["--max_samples", "64", "--snr_list", "0"] if args.smoke else []
                    run_cmd([py, "evaluate.py", "--checkpoint", ckpt_of(tag),
                             "--tune_llm", "--llm_model", bert_dir] + beam + tune_extra, env)
                evaluate(tag, channel, beam + ["--use_llm", "--llm_model", bert_dir],
                         f"beam{BEAM}_llm")

    # ---- Ablation plots ----
    run_cmd([py, "ablation.py",
             "--results_dir", ckpt_root,
             "--out_dir", result_dir], env)

    print(f"\nAll done!  Check {result_dir} for plots and tables.")


if __name__ == "__main__":
    main()

## 3. 数据：合并 Drive 上的分片并校验

In [ ]:
!cat "$PROJ"/data/sc_data.tar.gz.part_* | tar xz -C data
!ls -lh data

import hashlib
expected = {
    'sentences.pkl': '886bfe2761e394b20cbafe04080a74a6e1bcf083e792ae7d067a14bedc1b35c0',
    'test_data.pkl': 'd61ad3589e07914223411acab00f9ee753156cacdd93b1d7da549a88785847da',
    'train_data.pkl': '581faa9be369b97041914fac5a8d2bcc3a29a63c0790d1f2404c1122b5169bcc',
    'vocab.pkl': '2493197f50eb289eaf9e4b2c40968f46ca12efe166659c00e7761ce4e2f06ec8',
}
ok = True
for name, sha in expected.items():
    h = hashlib.sha256(open(f'data/{name}', 'rb').read()).hexdigest()
    ok &= h == sha
    print(f'{name:16s}', 'OK' if h == sha else 'MISMATCH')
assert ok, '数据和本地文件不一致'

## 4. 安装依赖、检查 GPU

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), '没有 GPU：请在“更改运行时类型”里选择 GPU'
print('bf16 (A100/L4 等):', torch.cuda.get_device_capability()[0] >= 8)

## 5. 快速自检（约 5–10 分钟）

每个模型只训练 20 个 batch、只评估 64 句，确认整条流程能跑通。输出写在 Colab 本地的 `smoke_checkpoints/`，不会影响 Drive 里的正式结果。

In [ ]:
os.environ['TQDM_DISABLE'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'
!python run_all.py --smoke 2>&1 | grep -E "Running:|\[Epoch|Results saved|Best lambda|All done|Error|Traceback" 

## 6. 正式训练和评估

全部结果写到 Drive。完整日志在 `checkpoints/run_all.log`；这里只显示每个 epoch 的汇总和各步骤的开始/结束。

In [ ]:
os.environ['SC_CKPT_DIR'] = f'{PROJ}/checkpoints'
os.environ['SC_RESULT_DIR'] = f'{PROJ}/results'
os.environ['TQDM_DISABLE'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'
os.makedirs(os.environ['SC_CKPT_DIR'], exist_ok=True)
!python run_all.py --amp 2>&1 | tee -a "$PROJ/checkpoints/run_all.log" | grep --line-buffered -E "Running:|Model:|\[Epoch|Training complete|Results saved|Best lambda|already|skipping|All done|Error|Traceback" 

## 7. 查看结果

In [ ]:
RESULTS = f'{PROJ}/results'
from IPython.display import Markdown, Image, display
import glob
display(Markdown(open(f'{RESULTS}/summary.md').read()))
for png in sorted(glob.glob(f'{RESULTS}/*BLEU-4*.png')) + [f'{RESULTS}/training_curves.png']:
    display(Image(png, width=700))